In [3]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import ast

MAIN_PATH = Path("../../data/processed/movies_metadata.csv")
DELIVERABLE = Path("../../data/features/08_revenue_features.csv")
#

`loading and inspecting supplemental dataframes`

In [4]:
#Read and inspect

df = pd.read_csv(MAIN_PATH)
 #Read and inspect supplemental paths 
nested = pd.read_csv("../../data/processed/nested_metadata.csv")
text = pd.read_csv("../../data/processed/movie_text.csv")
keywords = pd.read_csv("../../data/processed/keywords_clean.csv")
credits = pd.read_csv("../../data/processed/credits.csv")
print("nested_metadata") 
print(nested.head())
print("movie_text") 
print(text.head())
print("Movie_keywords")
print(keywords.head())
print("Movie_credits") 
print(credits.head())
print(df.shape)
print("Main Path") 
print(df.head())

nested_metadata
      id               original_title release_date                    genres  \
0    862                    Toy Story   1995-10-30   Animation|Comedy|Family   
1   8844                      Jumanji   1995-12-15  Adventure|Fantasy|Family   
2  15602             Grumpier Old Men   1995-12-22            Romance|Comedy   
3  31357            Waiting to Exhale   1995-12-22      Comedy|Drama|Romance   
4  11862  Father of the Bride Part II   1995-02-10                    Comedy   

            belongs_to_collection  \
0            Toy Story Collection   
1                             NaN   
2       Grumpy Old Men Collection   
3                             NaN   
4  Father of the Bride Collection   

                                production_companies  \
0                            Pixar Animation Studios   
1  TriStar Pictures|Teitler Film|Interscope Commu...   
2                        Warner Bros.|Lancaster Gate   
3             Twentieth Century Fox Film Corporation   


* `genres`, `production_companies`, `production_countries` and `spoken_languages` are in `nested_metadata.csv`.

* `director` and `top_cast` are in `credits.csv`, where `top_cast` is the `top_five_cast` column.

* `keywords` is `keyword_names_clean` in `keywords_clean.csv`.

* `title`, `original_title`, `overview`, `original_language` and `status` are in `movie_text.csv`.

In [5]:
#movies that dont have revenue
print("This many movies are missing a revenue value: ", df["revenue"].isnull().sum(), "; but this many are just 0:", (df["revenue"] == 0).sum())
#movies that dont have a budget
df['budget'] = pd.to_numeric(df['budget'], errors='coerce').astype('Int64')#budget values were being stored as string, converted to int
print("This many movies are missing a budget value: ", df["budget"].isnull().sum(), "; but this many are just 0:", (df["budget"] == 0).sum())

print("This many movies are missing a runtime value: ", df["runtime"].isnull().sum(), "; but this many are just 0:", (df["runtime"] == 0).sum())

This many movies are missing a revenue value:  6 ; but this many are just 0: 38052
This many movies are missing a budget value:  3 ; but this many are just 0: 36573
This many movies are missing a runtime value:  263 ; but this many are just 0: 1558


`cleaning rows that will be logged`

In [6]:
#replace missing/0 values with NaN to avoid removing all together, but as to not let them influence revenue data

# Replace 0 (sentinel values) with NaN
# Fill in the target value and replacement using .replace() 
df["revenue"] = df['revenue'].replace(0, np.nan)
df["budget"] = df['budget'].replace(0, np.nan)
df["runtime"] = df['runtime'].replace(0, np.nan)

# 3. Verify the transformation
print('Missing revenue count:', df['revenue'].isnull().sum())
print('Zero revenue count:', (df['revenue'] == 0).sum())
print('Missing budget count:', df['budget'].isnull().sum())
print('Zero budget count:', (df['budget'] == 0).sum())
print('Missing runtime count:', df['runtime'].isnull().sum())
print('Zero runtime count:', (df['runtime'] == 0).sum())


Missing revenue count: 38058
Zero revenue count: 0
Missing budget count: 36576
Zero budget count: 0
Missing runtime count: 1821
Zero runtime count: 0


`removing useless rows and merging datasets`

In [7]:
print("duplicated ids: ",df["id"].duplicated().sum())
print("empty ids: ",df["id"].isnull().sum())

df = df.dropna(subset=["id"])          #drop empty ids
df = df.drop_duplicates(subset = ["id"], keep = "first")  #drop duplicate ids

print("duplicated ids: ",df["id"].duplicated().sum())
print("empty ids: ",df["id"].isnull().sum())

#create final data set and merge in intial rows
full = (df.merge(nested, on="id", how = "left")
        .merge(text, on="id", how = "left")
        .merge(keywords, on="id", how = "left")
        .merge(credits, on="id", how = "left"))

#deal with duplicated columns (x,y) that got created somehow
full = full.drop(columns=["release_date_y", "original_title_y"]) #drop the duplicates
full = full.rename(columns={"release_date_x": "release_date", "original_title_x": "original_title"}) #remove _x from name

full.head()

duplicated ids:  32
empty ids:  3
duplicated ids:  0
empty ids:  0


,id,release_date,budget,revenue,runtime,popularity,original_title,genres,belongs_to_collection,production_companies,...,spoken_languages,title,overview,original_language,status,keyword_names_clean,director,top_five_cast,cast_size,crew_size
0,862.0,1995-10-30,30000000,373554033.0,81.0,21.946943,Toy Story,Animation|Comedy|Family,Toy Story Collection,Pixar Animation Studios,...,English,Toy Story,"Led by Woody, Andy's toys live happily in his ...",en,Released,"['jealousy', 'toy', 'boy', 'friendship', 'frie...",John Lasseter,"[""Tom Hanks"", ""Tim Allen"", ""Don Rickles"", ""Jim...",13.0,106.0
1,8844.0,1995-12-15,65000000,262797249.0,104.0,17.015539,Jumanji,Adventure|Fantasy|Family,NaN,TriStar Pictures|Teitler Film|Interscope Commu...,...,English|Français,Jumanji,When siblings Judy and Peter discover an encha...,en,Released,"['board game', 'disappearance', ""based on chil...",Joe Johnston,"[""Robin Williams"", ""Jonathan Hyde"", ""Kirsten D...",26.0,16.0
2,15602.0,1995-12-22,<NA>,NaN,101.0,11.712900,Grumpier Old Men,Romance|Comedy,Grumpy Old Men Collection,Warner Bros.|Lancaster Gate,...,English,Grumpier Old Men,A family wedding reignites the ancient feud be...,en,Released,"['fishing', 'best friend', 'duringcreditssting...",Howard Deutch,"[""Walter Matthau"", ""Jack Lemmon"", ""Ann-Margret...",7.0,4.0
3,31357.0,1995-12-22,16000000,81452156.0,127.0,3.859495,Waiting to Exhale,Comedy|Drama|Romance,NaN,Twentieth Century Fox Film Corporation,...,English,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",en,Released,"['based on novel', 'interracial relationship',...",Forest Whitaker,"[""Whitney Houston"", ""Angela Bassett"", ""Loretta...",10.0,10.0
4,11862.0,1995-02-10,<NA>,76578911.0,106.0,8.387519,Father of the Bride Part II,Comedy,Father of the Bride Collection,Sandollar Productions|Touchstone Pictures,...,English,Father of the Bride Part II,Just when George Banks has recovered from his ...,en,Released,"['baby', 'midlife crisis', 'confidence', 'agin...",Charles Shyer,"[""Steve Martin"", ""Diane Keaton"", ""Martin Short...",12.0,7.0


`Feature Engineering`

In [8]:
full = full.rename(columns={"keyword_names_clean": "keywords", "top_five_cast": "top_cast"}) #rename columns to match final dataset columns in issue description
pd.set_option('display.max_columns', None) #pandas starts to truncate columns

#create target_log_revenue feature
df["target_log_revenue"] = np.log1p(df["revenue"])
df.head()

#date
date = pd.to_datetime(full["release_date"]) #create date/time object from release_date
#creating date/time related features
full["release_year"] = date.dt.year.astype('Int64')
full["release_month"] = date.dt.month.astype('Int64')
full["release_decade"] = ((date.dt.year.astype('Int64') // 10) * 10) #integer division to remove last digit from year, then multiply by 10 to get decade

#financial and numeric
full["log_budget"] = np.log1p(full["budget"])
full["log_popularity"] = np.log1p(full["popularity"])
full.head()

#production
count = full["genres"].str.split("|").str.len().astype('Int64')  #split genre cells by seperator '|' and take length
full["genre_count"] = count

count = full["production_countries"].str.split("|").str.len().astype('Int64')  #split genre cells by seperator '|' and take length
full["country_count"] = count

count = full["production_companies"].str.split("|").str.len().astype('Int64')  #split genre cells by seperator '|' and take length
full["company_count"] = count

count = full["spoken_languages"].str.split("|").str.len().astype('Int64')  #split genre cells by seperator '|' and take length
full["language_count"] = count

value = full['belongs_to_collection'].isnull() #create bool value for each movie if 'belongs_to_collection' is null or not
full['has_collection'] = ~value  #flip bool values so NaN returns false instead of true


#credits
value = full['director'].isnull() #create bool value for each movie if 'director' is null or not
full['has_director'] = ~value  #flip bool values so NaN returns false instead of true

#keywords
count = full["keywords"].str.split(",").str.len().astype('Int64') #split list by ',' and take length
full["keyword_count"] = count

count = full["title"].str.split(" ").str.len().astype('Int64') #split column values by spaces between words and take length
full["title_word_count"] = count

count = full["overview"].str.split(" ").str.len().astype('Int64') #split column values by spaces between words and take length
full["overview_word_count"] = count

same_value = full['original_title'].eq(full["title"])  #compare orginal and current titles and return bool
full["title_differs_from_original"] = same_value

full.head()

,id,release_date,budget,revenue,runtime,popularity,original_title,genres,belongs_to_collection,production_companies,production_countries,spoken_languages,title,overview,original_language,status,keywords,director,top_cast,cast_size,crew_size,release_year,release_month,release_decade,log_budget,log_popularity,genre_count,country_count,company_count,language_count,has_collection,has_director,keyword_count,title_word_count,overview_word_count,title_differs_from_original
0,862.0,1995-10-30,30000000,373554033.0,81.0,21.946943,Toy Story,Animation|Comedy|Family,Toy Story Collection,Pixar Animation Studios,United States of America,English,Toy Story,"Led by Woody, Andy's toys live happily in his ...",en,Released,"['jealousy', 'toy', 'boy', 'friendship', 'frie...",John Lasseter,"[""Tom Hanks"", ""Tim Allen"", ""Don Rickles"", ""Jim...",13.0,106.0,1995,10,1990,17.216708,3.133185,3,1,1,1,True,True,9,2,50,True
1,8844.0,1995-12-15,65000000,262797249.0,104.0,17.015539,Jumanji,Adventure|Fantasy|Family,NaN,TriStar Pictures|Teitler Film|Interscope Commu...,United States of America,English|Français,Jumanji,When siblings Judy and Peter discover an encha...,en,Released,"['board game', 'disappearance', ""based on chil...",Joe Johnston,"[""Robin Williams"", ""Jonathan Hyde"", ""Kirsten D...",26.0,16.0,1995,12,1990,17.989898,2.891235,3,1,3,2,False,True,6,1,67,True
2,15602.0,1995-12-22,<NA>,NaN,101.0,11.712900,Grumpier Old Men,Romance|Comedy,Grumpy Old Men Collection,Warner Bros.|Lancaster Gate,United States of America,English,Grumpier Old Men,A family wedding reignites the ancient feud be...,en,Released,"['fishing', 'best friend', 'duringcreditssting...",Howard Deutch,"[""Walter Matthau"", ""Jack Lemmon"", ""Ann-Margret...",7.0,4.0,1995,12,1990,<NA>,2.542617,2,1,2,1,True,True,4,3,56,True
3,31357.0,1995-12-22,16000000,81452156.0,127.0,3.859495,Waiting to Exhale,Comedy|Drama|Romance,NaN,Twentieth Century Fox Film Corporation,United States of America,English,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",en,Released,"['based on novel', 'interracial relationship',...",Forest Whitaker,"[""Whitney Houston"", ""Angela Bassett"", ""Loretta...",10.0,10.0,1995,12,1990,16.588099,1.580935,3,1,1,1,False,True,5,3,45,True
4,11862.0,1995-02-10,<NA>,76578911.0,106.0,8.387519,Father of the Bride Part II,Comedy,Father of the Bride Collection,Sandollar Productions|Touchstone Pictures,United States of America,English,Father of the Bride Part II,Just when George Banks has recovered from his ...,en,Released,"['baby', 'midlife crisis', 'confidence', 'agin...",Charles Shyer,"[""Steve Martin"", ""Diane Keaton"", ""Martin Short...",12.0,7.0,1995,2,1990,<NA>,2.239381,1,1,2,1,True,True,9,6,59,True


In [10]:
deliverable_folder = Path("../../data/features")
deliverable_folder.mkdir(parents=True, exist_ok=True)
full.to_csv(DELIVERABLE, index = False)